In [80]:
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np

In [81]:

model = load_model('marriage_model.h5')
with open('serialized_object/label_encoder.pkl', 'rb') as file:
    label_encoder = pickle.load(file)
with open('serialized_object/onehot_encoder.pkl', 'rb') as file:
    onehot_encoder = pickle.load(file)
with open('serialized_object/scalar.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [82]:
# Create sample input object with natural/readable data
input_data = {
    'Age': 28,
    'Gender': 'Male',  # Natural format: Male/Female
    'Partner_Age': 28,
    'Partner_Gender': 'Female',
    'Years_Together': 7,
    'Age_Difference': 0,
    'Intercaste': 'No',  # Natural format: Yes/No
    'Are_They_Happy': 'Yes',
    'Want_To_Marry': 'Yes',
    'Compatibility_Score': 95,
    'Religion': 'Hindu',  # Natural format (will one-hot encode later)
    'Partner_Religion': 'Hindu',
    'Location_Type': 'Urban',  # Natural format (will one-hot encode later)
    'Partner_Location_Type': 'Urban',
}



In [83]:
input_df = pd.DataFrame([input_data])
input_df

,Age,Gender,Partner_Age,Partner_Gender,Years_Together,Age_Difference,Intercaste,Are_They_Happy,Want_To_Marry,Compatibility_Score,Religion,Partner_Religion,Location_Type,Partner_Location_Type
0,28,Male,28,Female,7,0,No,Yes,Yes,95,Hindu,Hindu,Urban,Urban


In [84]:
onehot_enocde = onehot_encoder.transform(input_df[['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type']])
onehot_enocde_df = pd.DataFrame(onehot_enocde.toarray(), columns=onehot_encoder.get_feature_names_out(['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type']))
onehot_enocde_df

,Religion_Buddhist,Religion_Christian,Religion_Hindu,Religion_Jain,Religion_Muslim,Religion_Parsi,Religion_Sikh,Partner_Religion_Buddhist,Partner_Religion_Christian,Partner_Religion_Hindu,Partner_Religion_Jain,Partner_Religion_Muslim,Partner_Religion_Parsi,Partner_Religion_Sikh,Location_Type_Rural,Location_Type_Semi-Urban,Location_Type_Urban,Partner_Location_Type_Rural,Partner_Location_Type_Semi-Urban,Partner_Location_Type_Urban
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [85]:
input_df['Gender'] = label_encoder.fit_transform(input_df['Gender'])
input_df['Partner_Gender'] = label_encoder.fit_transform(input_df['Partner_Gender'])
input_df['Intercaste'] = label_encoder.fit_transform(input_df['Intercaste'])
input_df['Are_They_Happy'] = label_encoder.fit_transform(input_df['Are_They_Happy'])
input_df['Want_To_Marry']= label_encoder.fit_transform(input_df['Want_To_Marry'])
input_df

,Age,Gender,Partner_Age,Partner_Gender,Years_Together,Age_Difference,Intercaste,Are_They_Happy,Want_To_Marry,Compatibility_Score,Religion,Partner_Religion,Location_Type,Partner_Location_Type
0,28,0,28,0,7,0,0,0,0,95,Hindu,Hindu,Urban,Urban


In [86]:
input_df = pd.concat([input_df.drop(['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type'], axis=1), onehot_enocde_df], axis=1)
input_df

,Age,Gender,Partner_Age,Partner_Gender,Years_Together,Age_Difference,Intercaste,Are_They_Happy,Want_To_Marry,Compatibility_Score,...,Partner_Religion_Jain,Partner_Religion_Muslim,Partner_Religion_Parsi,Partner_Religion_Sikh,Location_Type_Rural,Location_Type_Semi-Urban,Location_Type_Urban,Partner_Location_Type_Rural,Partner_Location_Type_Semi-Urban,Partner_Location_Type_Urban
0,28,0,28,0,7,0,0,0,0,95,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [87]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.77320265, -1.00175153, -0.75417499, -1.00828431, -0.180949  ,
        -1.42322611, -1.15251408, -1.38707484, -1.55103338,  0.66187593,
        -0.10113636, -0.18509419,  0.36246614, -0.065331  , -0.20709612,
        -0.05369627, -0.1588921 , -0.10050378, -0.18473268,  0.36202456,
        -0.07177326, -0.20445615, -0.0592646 , -0.15723042, -0.7077698 ,
        -0.7063777 ,  1.41408099, -0.69685592, -0.72014088,  1.41966693]])

In [88]:
prediction = model.predict(input_scaled)
print(f"Model Prediction (Probability of Getting Married): {prediction[0][0]:.4f}"  )
if prediction[0][0] >= 0.5:
    print("The model predicts that the couple is likely to get married.")
else:
    print("The model predicts that the couple is unlikely to get married.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Model Prediction (Probability of Getting Married): 0.9998
The model predicts that the couple is likely to get married.
